# Gaza building-damage classification — pretrained siamese pipeline

Fine-tunes an ImageNet-pretrained EfficientNetB0 (siamese, shared weights) on pre/post-event PlanetScope RGB patches (32×32 px, 3 m) to classify building damage.

**Key evaluation choices**

1. **Spatial holdout** — the city is cut into three contiguous bands along its principal axis (train / val / test), with a buffer strip at each boundary wider than one patch footprint (96 m). Neighboring patches overlap on the ground, so a random split would leak near-duplicate pixels into the test set and inflate scores.
2. **Threshold tuning** — the decision threshold is swept on the *validation* set (F1 / balanced accuracy / Youden's J) and the best one is applied — once — to the test set.
3. **No test leakage anywhere** — normalization percentiles and class weights are computed from the train split only; the test set is touched exactly once, at the end.

Other improvements over the original notebook: PR-AUC as the monitored metric (more informative than accuracy/ROC-AUC under class imbalance), model checkpointing to Drive, fixed seeds, a Lambda-free abs-difference layer (so the saved `.keras` file reloads cleanly), a spatial error map, and a saved `inference_config.json` with the threshold + normalization stats needed to reproduce predictions on new data.

In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - adjust paths below.")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
NPZ_PATH     = "/content/drive/MyDrive/War_Damage/datasets/Gaza_20240503_320a78e597.npz"
PARQUET_PATH = "/content/drive/MyDrive/War_Damage/datasets/Gaza_20240503_320a78e597.parquet"
OUT_DIR      = "/content/drive/MyDrive/War_Damage/models"

SEED         = 42
BATCH_SIZE   = 32
RESIZE_TO    = 128          # backbone input size (bilinear upsampling from 32px)

# Spatial split: contiguous bands along the city's principal axis
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20        # test gets the remainder (0.20)
PATCH_SIZE_M = 32 * 3                     # 32 px @ 3 m/px PlanetScope = 96 m
BUFFER_M     = PATCH_SIZE_M + 10          # dead zone at split boundaries -> no
                                          # patch can straddle two splits

# Threshold selection: metric optimized on the *validation* set
# options: "f1", "balanced_accuracy", "youden" (tpr - fpr)
THRESHOLD_METRIC = "f1"

import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── Imports & reproducibility ────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (precision_recall_curve, roc_curve, roc_auc_score,
                             average_precision_score, f1_score,
                             balanced_accuracy_score, confusion_matrix,
                             classification_report)

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1 · Load data and building coordinates
The parquet provides one row per building (same order as the `.npz`); its coordinates drive the spatial split.

In [ ]:
# ── Load imagery, labels and building coordinates ────────────────────────────
with np.load(NPZ_PATH) as data:
    images        = data["X"]                              # (N, 14, 32, 32)
    labels        = data["y"].astype(np.float32)           # (N,) binary 0/1
    channel_names = [n.strip() for n in data["channel_names"].tolist()]

buildings = pd.read_parquet(PARQUET_PATH)

# The parquet rows must align 1:1 with the npz rows - hard requirement for the
# spatial split. If this assert fires, the two files are not the same export.
assert len(buildings) == len(labels), (
    f"Row mismatch: parquet has {len(buildings)} rows, npz has {len(labels)}")

print(f"N = {len(labels)} | positive rate = {labels.mean():.3f}")
print(f"Channels: {channel_names}")
print(f"Parquet columns: {list(buildings.columns)}")


def extract_coords(df):
    """Return (x, y) arrays of building locations, trying common layouts."""
    cols = {c.lower(): c for c in df.columns}
    for cx, cy in [("longitude", "latitude"), ("lon", "lat"), ("lng", "lat"),
                   ("x", "y"), ("centroid_x", "centroid_y"),
                   ("center_x", "center_y")]:
        if cx in cols and cy in cols:
            return (df[cols[cx]].to_numpy(float),
                    df[cols[cy]].to_numpy(float), f"columns {cx}/{cy}")
    # geometry column (WKB bytes, WKT strings, or shapely objects)
    for gname in ("geometry", "geom", "wkb_geometry"):
        if gname in cols:
            from shapely import wkb, wkt
            def centroid(g):
                if isinstance(g, (bytes, bytearray)):
                    g = wkb.loads(bytes(g))
                elif isinstance(g, str):
                    g = wkt.loads(g)
                c = g.centroid
                return c.x, c.y
            xy = np.array([centroid(g) for g in df[cols[gname]]])
            return xy[:, 0], xy[:, 1], f"'{cols[gname]}' centroids"
    raise ValueError("No coordinate columns found. Available columns: "
                     f"{list(df.columns)} - add the right names to extract_coords().")


x_raw, y_raw, coord_src = extract_coords(buildings)
print(f"Coordinates from {coord_src}: "
      f"x∈[{x_raw.min():.4f}, {x_raw.max():.4f}], "
      f"y∈[{y_raw.min():.4f}, {y_raw.max():.4f}]")

# Convert lon/lat degrees to approximate local meters (equirectangular).
# If the values are already projected (magnitudes > 360), keep them as-is.
if np.abs(x_raw).max() <= 360 and np.abs(y_raw).max() <= 90:
    lat0 = np.deg2rad(y_raw.mean())
    x_m = (x_raw - x_raw.mean()) * 111_320 * np.cos(lat0)
    y_m = (y_raw - y_raw.mean()) * 110_540
else:
    x_m, y_m = x_raw - x_raw.mean(), y_raw - y_raw.mean()

## 2 · Spatial holdout split

In [ ]:
# ── Spatial holdout split ────────────────────────────────────────────────────
# Neighboring 32x32 patches overlap on the ground, so a random split leaks
# nearly-identical pixels between train and test. Instead we cut the city into
# three contiguous bands along its principal (longest) axis, and drop a buffer
# strip at each boundary so no patch can appear (even partially) in two splits.

XY = np.stack([x_m, y_m], axis=1)
XY_c = XY - XY.mean(axis=0)
_, _, Vt = np.linalg.svd(XY_c, full_matrices=False)
t = XY_c @ Vt[0]                       # position along the principal axis (m)

b1 = np.quantile(t, TRAIN_FRAC)                    # train | val boundary
b2 = np.quantile(t, TRAIN_FRAC + VAL_FRAC)         # val | test boundary
half = BUFFER_M / 2.0

split = np.full(len(t), "buffer", dtype=object)
split[t <  b1 - half] = "train"
split[(t >= b1 + half) & (t < b2 - half)] = "val"
split[t >= b2 + half] = "test"

idx_train = np.where(split == "train")[0]
idx_val   = np.where(split == "val")[0]
idx_test  = np.where(split == "test")[0]

print(f"{'split':<8}{'n':>6}{'pos rate':>10}")
for name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test),
                  ("buffer", np.where(split == "buffer")[0])]:
    pr = labels[idx].mean() if len(idx) else float("nan")
    print(f"{name:<8}{len(idx):>6}{pr:>10.3f}")

for name, idx in [("val", idx_val), ("test", idx_test)]:
    if len(np.unique(labels[idx])) < 2:
        print(f"WARNING: {name} split contains a single class - "
              "swap band order or adjust fractions.")

# Sanity check: minimum distance between any train and any test building
from scipy.spatial import cKDTree
d, _ = cKDTree(XY[idx_train]).query(XY[idx_test], k=1)
print(f"\nMin train↔test distance: {d.min():.0f} m "
      f"(patch footprint {PATCH_SIZE_M} m -> no overlap possible: {d.min() > PATCH_SIZE_M})")

In [ ]:
# ── Visualize the split ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)

colors = {"train": "tab:blue", "val": "tab:orange",
          "test": "tab:green", "buffer": "lightgray"}
for name, c in colors.items():
    m = split == name
    axes[0].scatter(x_m[m], y_m[m], s=4, c=c, label=f"{name} ({m.sum()})")
axes[0].legend(markerscale=3)
axes[0].set_title("Spatial split (contiguous bands + buffer)")

axes[1].scatter(x_m[labels == 0], y_m[labels == 0], s=4, c="silver", label="intact")
axes[1].scatter(x_m[labels == 1], y_m[labels == 1], s=4, c="crimson", label="damaged")
axes[1].legend(markerscale=3)
axes[1].set_title("Damage labels")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
plt.tight_layout()
plt.show()

## 3 · Preprocessing and `tf.data` pipelines

In [ ]:
# ── Extract pre/post RGB and normalize (stats from TRAIN split only) ─────────
pre_ch  = [channel_names.index(c) for c in ["ps_pre_R",  "ps_pre_G",  "ps_pre_B"]]
post_ch = [channel_names.index(c) for c in ["ps_post_R", "ps_post_G", "ps_post_B"]]

# channels-last float32, R-G-B order
pre_imgs  = np.transpose(images[:, pre_ch,  :, :], (0, 2, 3, 1)).astype(np.float32)
post_imgs = np.transpose(images[:, post_ch, :, :], (0, 2, 3, 1)).astype(np.float32)
pre_imgs  = np.nan_to_num(pre_imgs,  nan=0.0, posinf=0.0, neginf=0.0)
post_imgs = np.nan_to_num(post_imgs, nan=0.0, posinf=0.0, neginf=0.0)

# Robust per-channel 2-98 percentile clip. Computing the percentiles on the
# train split only avoids leaking val/test statistics into preprocessing.
train_stack = np.concatenate([pre_imgs[idx_train], post_imgs[idx_train]], axis=0)
lo = np.percentile(train_stack, 2,  axis=(0, 1, 2)).astype(np.float32)
hi = np.percentile(train_stack, 98, axis=(0, 1, 2)).astype(np.float32)
del train_stack
print(f"Per-channel clip range:\n  lo = {lo}\n  hi = {hi}")

def normalize(x):
    x = np.clip(x, lo, hi)
    return (x - lo) / np.maximum(hi - lo, 1e-6)

pre_imgs, post_imgs = normalize(pre_imgs), normalize(post_imgs)

In [ ]:
# ── tf.data pipelines with pair-consistent augmentation ──────────────────────
def augment_pair(inputs, label):
    """Identical geometric transform for pre and post; mild independent
    brightness jitter to mimic acquisition differences."""
    pre, post = inputs
    both = tf.concat([pre, post], axis=-1)                # (32, 32, 6)
    both = tf.image.random_flip_left_right(both)
    both = tf.image.random_flip_up_down(both)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    both = tf.image.rot90(both, k)
    pre, post = both[..., :3], both[..., 3:]
    pre  = tf.clip_by_value(pre  + tf.random.uniform([], -0.05, 0.05), 0.0, 1.0)
    post = tf.clip_by_value(post + tf.random.uniform([], -0.05, 0.05), 0.0, 1.0)
    return (pre, post), label


def make_ds(indices, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        ((pre_imgs[indices], post_imgs[indices]), labels[indices]))
    if training:
        ds = ds.shuffle(len(indices), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(idx_train, training=True)
val_ds   = make_ds(idx_val)      # no shuffle -> prediction order == idx order
test_ds  = make_ds(idx_test)

# Class weights from the train split
n_pos = labels[idx_train].sum()
n_neg = len(idx_train) - n_pos
class_weight = {0: len(idx_train) / (2.0 * n_neg),
                1: len(idx_train) / (2.0 * n_pos)}
print(f"Class weights: {class_weight}")

## 4 · Model

In [ ]:
# ── Siamese model: shared ImageNet-pretrained EfficientNetB0 encoder ─────────
INPUT_SHAPE = (32, 32, 3)

def build_model():
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(RESIZE_TO, RESIZE_TO, 3), pooling="avg")
    backbone.trainable = False                      # phase 1: frozen

    # Shared encoder. EfficientNet expects raw [0, 255] input (its own
    # normalization layers are built in), hence the Rescaling(255).
    inp = layers.Input(INPUT_SHAPE)
    x = layers.Resizing(RESIZE_TO, RESIZE_TO, interpolation="bilinear")(inp)
    x = layers.Rescaling(255.0)(x)
    x = backbone(x)
    encoder = models.Model(inp, x, name="shared_encoder")

    pre_in  = layers.Input(INPUT_SHAPE, name="pre_image")
    post_in = layers.Input(INPUT_SHAPE, name="post_image")
    f_pre, f_post = encoder(pre_in), encoder(post_in)

    # |f_pre - f_post| built from serializable layers only (no Lambda,
    # so the saved .keras file reloads without safe_mode workarounds):
    # abs(d) = relu(d) + relu(-d)
    d = layers.Subtract()([f_pre, f_post])
    abs_d = layers.Add()([layers.ReLU()(d),
                          layers.ReLU()(layers.Rescaling(-1.0)(d))])

    x = layers.Concatenate()([f_pre, f_post, abs_d])
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid", name="damage")(x)

    return models.Model([pre_in, post_in], out, name="siamese_damage"), backbone


model, backbone = build_model()
model.summary(line_length=100)

# PR-AUC is the headline metric: with class imbalance it is far more
# informative than accuracy or ROC-AUC, and it is threshold-independent.
METRICS = [tf.keras.metrics.AUC(name="auc_pr", curve="PR"),
           tf.keras.metrics.AUC(name="auc_roc"),
           tf.keras.metrics.BinaryAccuracy(name="acc"),
           tf.keras.metrics.Precision(name="precision"),
           tf.keras.metrics.Recall(name="recall")]

CKPT_PATH = f"{OUT_DIR}/siamese_effnetb0_gaza_best.keras"

def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_auc_pr", mode="max",
                                         patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc_pr", mode="max",
                                             factor=0.5, patience=4, min_lr=1e-6),
        tf.keras.callbacks.ModelCheckpoint(CKPT_PATH, monitor="val_auc_pr",
                                           mode="max", save_best_only=True),
    ]

## 5 · Training (two phases)

In [ ]:
# ── Phase 1: train the head with a frozen backbone ───────────────────────────
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=METRICS)
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=20,
                  class_weight=class_weight, callbacks=make_callbacks())

In [ ]:
# ── Phase 2: unfreeze the top of the backbone, fine-tune at low LR ───────────
backbone.trainable = True
for layer in backbone.layers[:-60]:              # keep early layers frozen
    layer.trainable = False
for layer in backbone.layers:                    # keep BatchNorm frozen
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy", metrics=METRICS)
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=30,
                  class_weight=class_weight, callbacks=make_callbacks())

## 6 · Threshold selection (validation set)

In [ ]:
# ── Threshold selection on the VALIDATION set ────────────────────────────────
# The 0.5 default is arbitrary, especially with class weighting. We sweep
# thresholds on the val set, pick the one maximizing THRESHOLD_METRIC, and
# only then touch the test set (once, with that fixed threshold).
val_probs = model.predict(val_ds, verbose=0).ravel()
val_y = labels[idx_val]

grid = np.linspace(0.01, 0.99, 197)
def sweep(metric_fn):
    return np.array([metric_fn(val_y, (val_probs >= th).astype(int)) for th in grid])

scores = {
    "f1":                sweep(lambda y, p: f1_score(y, p, zero_division=0)),
    "balanced_accuracy": sweep(balanced_accuracy_score),
}
fpr, tpr, roc_th = roc_curve(val_y, val_probs)
# map Youden's J onto the same grid for a uniform interface
youden_on_grid = np.interp(grid, roc_th[::-1], (tpr - fpr)[::-1])
scores["youden"] = youden_on_grid

best_thr = float(grid[np.argmax(scores[THRESHOLD_METRIC])])
print(f"Best threshold by {THRESHOLD_METRIC} (on val): {best_thr:.3f}")
for name, s in scores.items():
    th = grid[np.argmax(s)]
    print(f"  argmax {name:<18}-> thr={th:.3f}, score={s.max():.3f}")

plt.figure(figsize=(7, 4.5))
for name, s in scores.items():
    plt.plot(grid, s, label=name)
plt.axvline(best_thr, ls="--", c="k", label=f"chosen ({best_thr:.3f})")
plt.axvline(0.5, ls=":", c="gray", label="default 0.5")
plt.xlabel("threshold"); plt.ylabel("val score"); plt.legend()
plt.title("Threshold sweep (validation set)")
plt.tight_layout(); plt.show()

## 7 · Test evaluation (spatially held out, evaluated once)

In [ ]:
# ── Final evaluation on the spatially held-out TEST set ──────────────────────
test_probs = model.predict(test_ds, verbose=0).ravel()
test_y = labels[idx_test]

print(f"Test ROC-AUC: {roc_auc_score(test_y, test_probs):.4f}")
print(f"Test PR-AUC:  {average_precision_score(test_y, test_probs):.4f}")

for thr, tag in [(best_thr, f"chosen thr={best_thr:.3f}"), (0.5, "default thr=0.500")]:
    pred = (test_probs >= thr).astype(int)
    print(f"\n─── {tag} ───")
    print(f"F1: {f1_score(test_y, pred):.4f} | "
          f"balanced acc: {balanced_accuracy_score(test_y, pred):.4f}")
    print("Confusion matrix [[TN FP][FN TP]]:")
    print(confusion_matrix(test_y, pred))
    print(classification_report(test_y, pred,
                                target_names=["intact", "damaged"], digits=3))

# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(test_y, test_probs)
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], ls=":", c="gray")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("Test ROC")
prec, rec, _ = precision_recall_curve(test_y, test_probs)
axes[1].plot(rec, prec)
axes[1].axhline(test_y.mean(), ls=":", c="gray", label="prevalence")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Test Precision-Recall"); axes[1].legend()
plt.tight_layout(); plt.show()

# Where do the errors sit on the map? Spatial error clusters usually mean the
# model is picking up neighborhood context rather than building damage.
pred = (test_probs >= best_thr).astype(int)
err = pred != test_y.astype(int)
plt.figure(figsize=(6, 6))
plt.scatter(x_m[idx_test][~err], y_m[idx_test][~err], s=6, c="silver", label="correct")
plt.scatter(x_m[idx_test][err],  y_m[idx_test][err],  s=10, c="crimson", label="error")
plt.gca().set_aspect("equal"); plt.legend(markerscale=2)
plt.title("Test-set errors in space"); plt.tight_layout(); plt.show()

## 8 · Save model + inference config

In [ ]:
# ── Save model + everything needed to reproduce inference ────────────────────
MODEL_PATH = f"{OUT_DIR}/siamese_effnetb0_gaza_final.keras"
model.save(MODEL_PATH)

# The threshold and normalization stats are part of the model contract:
# without them, predictions on new data are not reproducible.
inference_config = {
    "model_path": MODEL_PATH,
    "decision_threshold": best_thr,
    "threshold_metric": THRESHOLD_METRIC,
    "norm_lo": lo.tolist(),
    "norm_hi": hi.tolist(),
    "pre_channels":  ["ps_pre_R", "ps_pre_G", "ps_pre_B"],
    "post_channels": ["ps_post_R", "ps_post_G", "ps_post_B"],
    "resize_to": RESIZE_TO,
    "split": {"type": "spatial_bands_principal_axis",
              "train_frac": TRAIN_FRAC, "val_frac": VAL_FRAC,
              "buffer_m": BUFFER_M, "seed": SEED},
}
with open(f"{OUT_DIR}/inference_config.json", "w") as f:
    json.dump(inference_config, f, indent=2)

print(f"Saved model -> {MODEL_PATH}")
print(f"Saved inference config -> {OUT_DIR}/inference_config.json")
print(json.dumps(inference_config, indent=2))